In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT5 project root (paths.py). Run Jupyter with cwd GPT5 or GPT5/notebooks."
    )
import paths

In [1]:
import openai
import pandas as pd

# Load the CSV file
df = pd.read_csv(paths.DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv")

# Display the first few rows after removing duplicates
print(df)
# df.columns
# print(len(df))

      Origin data_source_df3  \
0     ID0002            jama   
1     ID0003        medxpert   
2     ID0007      medbullets   
3     ID0009            jama   
4     ID0010        medxpert   
...      ...             ...   
1295  ID1995            mmlu   
1296  ID1996            mmlu   
1297  ID1997        medxpert   
1298  ID1998      medbullets   
1299  ID1999        medxpert   

                                        Patient_Profile  \
0     1. A woman in her 60s with a history of hyperl...   
1     1. A 20-year-old woman comes to the primary ca...   
2     1. A 72-year-old man presents to his primary c...   
3     1. A woman in her 30s presented with multiple ...   
4     1. A 17-year-old high school student accidenta...   
...                                                 ...   
1295  1. A 17-year-old girl is brought to the emerge...   
1296  1. A 68-year-old female presents to the emerge...   
1297  1. A 27-year-old woman presents with a 4-month...   
1298  1. A 6-month-old gi

In [2]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-5 with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-5",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"

In [3]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.PREDICTIONS / "[SR]_gpt5_predictions_on_72B_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["Qwen72B_High_Relevance"], row["question_options_x"])
    
    result_row = row.to_dict()
    result_row["gpt5_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


100%|█████████████████████████████████████| 1300/1300 [6:01:08<00:00, 16.67s/it]


In [21]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.PREDICTIONS / "[SR]_gpt5_predictions_on_72B_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the XML format - UPDATED to handle brackets
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        # Try to match with optional brackets around the letter
        match = re.search(r"<answer>Option\s+\[?([A-J])\]?</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["GPT5_on_72B_SR"]

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_df3"]

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")
    

# Save the updated dataframe with the new columns back to CSV
df["gpt_letter"].to_csv(output_path, index=False)
print(f"\nUpdated data saved to '{output_path}'")

KeyError: 'answer_df3'

In [19]:
# Load the model predictions
output_path = paths.PREDICTIONS / "[SR]_gpt5_predictions_on_72B_removed.csv"
df = pd.read_csv(output_path)
df

,QA_ID,context,question_options,answer_df3,data_source_corr,Origin,70B_sentence_ids,70B_After_Removal,human_sentence_ids,Low_Irr_70B,Extra_Context_Minus_70B,gpt_direct_prediction,72B_Sentence_Contents,72B_Low_Irr,gpt5_direct_prediction,GPT5_on_72B_SR
0,Merge Q229,There were no palpable adnexal masses or orbit...,What Would You Do Next?\n\nA: Perform left upp...,D,jama,ID0002,4. 6. 7. 8. 9.,4. External examination was notable for left u...,"1, 2, 4, 6, 11",External examination was notable for left uppe...,There were no palpable adnexal masses or orbit...,<answer>Option B</answer>,There were no palpable adnexal masses or orbit...,NaN,<answer>Option D</answer>,B
1,Merge Q2235,Sputum culture and Gram stain are negative. Ch...,Which of the following components is essential...,G,medxpert,ID0003,1. 2. 5. 6. 7.,1. A 20-year-old woman comes to the primary ca...,"1, 2, 4, 5, 6","On examination, her temperature is 101.5 ºF, b...",Sputum culture and Gram stain are negative Che...,<answer>Option D</answer>,Sputum culture and Gram stain are negative. | ...,NaN,<answer>Option G</answer>,G
2,Merge Q1063,The patient has lost 12 pounds recently and ha...,Which of the following diagnostic tests would ...,B,medbullets,ID0007,1. 4. 5. 6. 7.,1. A 72-year-old man presents to his primary c...,"2, 3, 4, 5, 7","His temperature is 99.5°F (37.5°C), blood pres...",The patient has lost 12 pounds recently and ha...,<answer>Option B</answer>,The patient has lost 12 pounds recently and ha...,NaN,<answer>Option B</answer>,B
3,Merge Q596,The pain associated with these papules worsene...,What Is Your Diagnosis?\n\nA: Blue rubber bleb...,D,jama,ID0009,1. 3. 5. 6. 7. 8.,1. A woman in her 30s presented with multiple ...,"2, 3, 4, 5, 6",The patient underwent duplex ultrasonography o...,The pain associated with these papules worsene...,<answer>Option D</answer>,The pain associated with these papules worsene...,A woman in her 30s presented with multiple dis...,<answer>Option D</answer>,D
4,Merge Q3192,Vital signs show a temperature of 98.0°F (36.6...,What is the proper method for transporting the...,H,medxpert,ID0010,4. 7.,4. Vital signs show a temperature of 98 7. The...,"1, 7",Vital signs show a temperature of 98.0°F (36.6...,Vital signs show a temperature of 98.0°F (36.6...,<answer>Option H</answer>,Vital signs show a temperature of 98.0°F (36.6...,NaN,<answer>Option H</answer>,H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,Merge Q1074,The patient’s father brought a 4-week follow-u...,Which of the structures labeled in Figure A wo...,A,medbullets,ID1998,1. 2. 5. 6. 7. 8.,1. A 6-month-old girl is brought to the pediat...,"1, 2, 3, 4, 5, 6",She often babbles but sometimes does make iden...,The patient’s father brought a 4-week follow-u...,"I'm sorry, but I cannot determine the most app...",The patient’s father brought a 4-week follow-u...,NaN,<answer>Option A</answer>,D
1296,Merge Q1555,"Her history reveals self-reported ""mood swings...",What is the most likely diagnosis?\n\nA. Preme...,C,medxpert,ID1999,3. 4.,3. She is not on any medications and consumes ...,"1, 2, 5","Her history reveals self-reported ""mood swings...","Her history reveals self-reported ""mood swings...","I'm sorry, but I need the context or details o...","Her history reveals self-reported ""mood swings...",NaN,<answer>Option E</answer>,D
1297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B
1298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A


In [5]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr_trainee"].unique():
    source_df = df[df["data_source_corr_trainee"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")

Standard Deviation of Accuracy Across Data Sources: 0.1299


In [ ]:
# --- Physician Origin subset recalculation ---
from pathlib import Path
import pandas as pd
import re
import numpy as np

physician_csv = Path(r"/home/yuexing/NeuRIPS25/Physician_Labels/Mar2_2026_Data/933_Clinician_Student_Majority_Vote.csv")
physician_origins = set(
    pd.read_csv(physician_csv, usecols=['Origin'])['Origin'].astype(str).str.strip()
)

if 'df' in locals() and isinstance(df, pd.DataFrame):
    df_eval = df.copy()
elif 'output_path' in locals():
    df_eval = pd.read_csv(output_path)
else:
    raise RuntimeError('Could not find dataframe `df` or `output_path` in this notebook state.')

if 'Origin' not in df_eval.columns:
    raise KeyError('`Origin` column is missing from evaluation dataframe.')

df_eval = df_eval[df_eval['Origin'].astype(str).str.strip().isin(physician_origins)].copy()
print(f"Physician-Origin subset rows: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError('No overlapping Origin IDs found with physician CSV.')

# Build gpt_letter when not already present
if 'gpt_letter' not in df_eval.columns:
    pred_col = None
    for c in ['gpt5_direct_prediction', 'gpt4o_direct_prediction', 'majority_vote', 'GPT5_on_72B_SR']:
        if c in df_eval.columns:
            pred_col = c
            break
    if pred_col is None:
        raise KeyError('No supported prediction column found to derive `gpt_letter`.')

    def extract_letter(x):
        if not isinstance(x, str):
            return None
        m = re.search(r'Option\s*\[?([A-J])\]?|^\s*([A-J])\s*$', str(x).strip(), flags=re.IGNORECASE)
        if m:
            return (m.group(1) or m.group(2)).upper()
        return None

    if pred_col == 'GPT5_on_72B_SR':
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    elif pred_col == 'majority_vote' and 'answer_corr' in df_eval.columns:
        # For this notebook style majority_vote is often already a letter
        df_eval['gpt_letter'] = df_eval[pred_col].astype(str).str.strip().str.upper()
    else:
        df_eval['gpt_letter'] = df_eval[pred_col].apply(extract_letter)

# Build answer_letter
if 'answer_letter' not in df_eval.columns:
    if 'answer_corr' in df_eval.columns:
        df_eval['answer_letter'] = df_eval['answer_corr'].astype(str).str.strip().str.upper()
    else:
        raise KeyError('No `answer_corr` column available to build `answer_letter`.')

# Match + binary columns
if 'gpt_letter_match' not in df_eval.columns:
    df_eval['gpt_letter_match'] = np.where(
        df_eval['gpt_letter'] == df_eval['answer_letter'],
        'Correct',
        'Incorrect'
    )

df_eval['gpt_letter_binary'] = (df_eval['gpt_letter_match'] == 'Correct').astype(int)

correct_count = int(df_eval['gpt_letter_binary'].sum())
total_count = int(df_eval['gpt_letter_binary'].notna().sum())
accuracy = correct_count / total_count if total_count > 0 else 0.0

print('\n=== Physician-Origin Recalculation ===')
print(f"Correct Predictions: {correct_count}")
print(f"Total Predictions: {total_count}")
print(f"Accuracy: {accuracy:.2%}")

source_col = None
for c in ['data_source_df3', 'data_source_corr', 'data_source_corr_trainee']:
    if c in df_eval.columns:
        source_col = c
        break

if source_col:
    print(f"\nPer-data-source stats ({source_col}):")
    for source in df_eval[source_col].dropna().unique():
        source_df = df_eval[df_eval[source_col] == source]
        c = int(source_df['gpt_letter_binary'].sum())
        t = int(source_df['gpt_letter_binary'].notna().sum())
        acc = c / t if t > 0 else 0.0
        std = source_df['gpt_letter_binary'].std(ddof=1) if t > 1 else float('nan')
        print(f"  {source}: Correct={c}, Total={t}, Accuracy={acc:.2%}, Std={std:.4f}")
